In [ ]:
# Output QA — SEBI Disclosure Pipeline

This notebook audits the output of the Java Spring Boot pipeline.
Goal: verify the results.json is accurate, complete, and correctly
formatted before submission.

Checks performed:
1. Schema validation — all required fields present and typed correctly
2. Confidence distribution — are low-confidence results plausible?
3. Manual spot-check — 10 random extractions verified against source PDF
4. Failure audit — what failed and why?
5. Edge case review — multi-change docs, null dates, CFO exclusions

In [ ]:
import json
import pandas as pd
import pdfplumber
from pathlib import Path

OUTPUT_FILE = Path("../output/results.json")
INPUT_DIR   = Path("../input")

with open(OUTPUT_FILE) as f:
    data = json.load(f)

extractions = data["extractions"]
summary     = data["summary"]
df          = pd.DataFrame(extractions)

print("=== PIPELINE SUMMARY ===")
print(f"Total documents processed      : {summary['total_documents_processed']}")
print(f"Director change documents      : {summary['director_change_documents_identified']}")
print(f"Total director changes         : {summary['total_director_changes_extracted']}")
print(f"Documents failed               : {len(summary['documents_that_failed_processing'])}")
print(f"\nFailed documents:")
for f in summary["documents_that_failed_processing"]:
    print(f"  {f}")

In [ ]:
print("=== SCHEMA VALIDATION ===\n")

required_fields = [
    "source_filename", "company_name", "stock_ticker",
    "director_name", "change_type", "effective_date",
    "reason_stated", "extraction_confidence"
]

valid_change_types  = {"appointment", "resignation", "removal"}
valid_confidences   = {"high", "medium", "low"}

schema_errors = []

for i, record in enumerate(extractions):
    for field in required_fields:
        if field not in record:
            schema_errors.append(f"Record {i} ({record.get('source_filename')}) missing field: {field}")

    if record.get("change_type") not in valid_change_types:
        schema_errors.append(
            f"Record {i} invalid change_type: {record.get('change_type')}")

    if record.get("extraction_confidence") not in valid_confidences:
        schema_errors.append(
            f"Record {i} invalid confidence: {record.get('extraction_confidence')}")

    # effective_date must be YYYY-MM-DD or null
    date_val = record.get("effective_date")
    if date_val is not None:
        try:
            from datetime import datetime
            datetime.strptime(date_val, "%Y-%m-%d")
        except ValueError:
            schema_errors.append(
                f"Record {i} invalid date format: {date_val}")

if schema_errors:
    print(f"SCHEMA ERRORS FOUND ({len(schema_errors)}):")
    for err in schema_errors:
        print(f"  {err}")
else:
    print("All records pass schema validation.")

In [ ]:
print("=== CONFIDENCE DISTRIBUTION ===\n")

conf_counts = df["extraction_confidence"].value_counts()
print(conf_counts.to_string())

print("\nLow confidence extractions (review manually):")
low_conf = df[df["extraction_confidence"] == "low"]
if low_conf.empty:
    print("  None — good.")
else:
    for _, row in low_conf.iterrows():
        print(f"  {row['source_filename']} — {row['director_name']} — {row['change_type']}")

In [ ]:
print("=== NULL FIELD ANALYSIS ===\n")

null_dates    = df[df["effective_date"].isna()]
null_tickers  = df[df["stock_ticker"].isna()]
null_reasons  = df[df["reason_stated"].isna()]
null_names    = df[df["director_name"].isna()]

print(f"Null effective_date  : {len(null_dates)} records")
print(f"Null stock_ticker    : {len(null_tickers)} records")
print(f"Null reason_stated   : {len(null_reasons)} records")
print(f"Null director_name   : {len(null_names)} records")

if not null_names.empty:
    print("\nWARNING — null director_name is a likely extraction failure:")
    for _, row in null_names.iterrows():
        print(f"  {row['source_filename']}")

In [ ]:
print("=== MULTI-CHANGE DOCUMENTS ===\n")

multi = df.groupby("source_filename").size()
multi_change_docs = multi[multi > 1]

if multi_change_docs.empty:
    print("No documents produced more than one extraction.")
    print("NOTE: manually verify whether multi-change PDFs exist in input/")
else:
    print(f"Documents with multiple extractions: {len(multi_change_docs)}")
    for filename, count in multi_change_docs.items():
        print(f"  {filename}: {count} changes")
        subset = df[df["source_filename"] == filename]
        for _, row in subset.iterrows():
            print(f"    - {row['director_name']} ({row['change_type']})")

In [ ]:
print("=== MANUAL SPOT-CHECK — 10 RANDOM EXTRACTIONS ===\n")
print("For each extraction below, open the source PDF and verify manually.\n")

sample = df.sample(min(10, len(df)), random_state=42)

def get_first_500_chars(filename: str) -> str:
    path = INPUT_DIR / filename
    try:
        with pdfplumber.open(path) as pdf:
            return (pdf.pages[0].extract_text() or "")[:500]
    except Exception as e:
        return f"[Could not read PDF: {e}]"

for _, row in sample.iterrows():
    print(f"FILE      : {row['source_filename']}")
    print(f"COMPANY   : {row['company_name']}")
    print(f"DIRECTOR  : {row['director_name']}")
    print(f"CHANGE    : {row['change_type']}")
    print(f"DATE      : {row['effective_date']}")
    print(f"CONFIDENCE: {row['extraction_confidence']}")
    print(f"\nPDF EXCERPT (first 500 chars of page 1):")
    print(get_first_500_chars(row["source_filename"]))
    print(f"\n{'='*60}\n")

In [ ]:
print("=== FINAL QA SUMMARY ===\n")

total         = summary["total_documents_processed"]
dc_docs       = summary["director_change_documents_identified"]
total_changes = summary["total_director_changes_extracted"]
failed        = len(summary["documents_that_failed_processing"])
schema_ok     = len(schema_errors) == 0
null_name_ok  = null_names.empty

print(f"Documents processed     : {total}")
print(f"Director change docs    : {dc_docs} ({dc_docs/total*100:.1f}%)")
print(f"Total changes extracted : {total_changes}")
print(f"Failed documents        : {failed}")
print(f"Schema valid            : {'YES' if schema_ok else 'NO — see errors above'}")
print(f"No null director names  : {'YES' if null_name_ok else 'NO — review above'}")
print(f"Low confidence count    : {len(low_conf)}")
print()
print("Pipeline is ready for submission." if (schema_ok and null_name_ok)
      else "Pipeline has issues — review errors above before submitting.")